# Initialize Database

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    path = Path.cwd()
    while path != path.parent:
        if (path / "pyproject.toml").exists():
            return path
        path = path.parent
    raise RuntimeError("Could not find project root")

database = str(find_project_root() / "data" / "autogc.db")

## EQ setup

In [10]:
import logging

logging.basicConfig(level = logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
date = "2025-03-01 00:00:00"
from autogc_validation.database.models import *
from autogc_validation.database.operations import insert, get_table, get_active_canister_concentrations
from autogc_validation.database.enums import *
site_eq = Site(site_id=490353015, name_short = 'EQ', name_long = "Utah Technical Center", lat = 40.7770994964404, long = -111.9450044213331, date_started="2023-10-01 00:00:00")
insert(database, site_eq)
cvs1 = PrimaryCanister(primary_canister_id= "CC524930-0626", canister_type="CVS", expiration_date="2026-06-01 00:00:00")
insert(database, cvs1)
cvs1_dr = .00189
cvs1_concentrations = {
    # Channel A
    "ETHANE": 0.525,
    "PROPANE": 0.34,
    "N-BUTANE": 0.253,
    "ACETYLENE": 0.525,
    "N-PENTANE": 0.204,
    "1,3-BUTADIENE": 0.263,
    "2-METHYLPENTANE": 0.17,
    "1-HEXENE": 0.17,

    # Channel B
    "N-HEXANE": 0.167,
    "BENZENE": 0.175,
    "TOLUENE": 0.146,
    "M&P-XYLENE": 0.131,
    "N-PROPYLBENZENE": 0.116,
    "1,2,4-TRI-M-BENZENE": 0.113,
    "P-DIETHYLBENZENE": 0.102,
}
for compound, concentration in cvs1_concentrations.items():
    aqs = name_to_aqs(compound.capitalize())
    p = CanisterConcentration(primary_canister_id = cvs1.primary_canister_id, aqs_code = aqs, concentration=concentration, units = "ppmv", canister_type="CVS")
    insert(database, p)
cvs1_concentrations = get_table(database, CanisterConcentration.__tablename__)
cvs1_concentrations
eq_cvs1 = SiteCanister(site_canister_id="3667", site_id=site_eq.site_id, primary_canister_id=cvs1.primary_canister_id, dilution_ratio=cvs1_dr,date_on="2024-09-07 00:00:00", date_off="2025-04-16 08:59:59")
insert(database, eq_cvs1)
eq_df = get_table(database, SiteCanister.__tablename__)
cvs_conc = get_active_canister_concentrations(database, site_eq.site_id, "CVS", date, output_unit = "ppbc")
cvs_conc["aqs_code"] = [aqs_to_name(code) for code in cvs_conc["aqs_code"]]
cvs_conc


## Backup database

Run this cell after making any changes to dump the database to `data/autogc.sql`.
Then commit the file to save a restorable snapshot.

In [ ]:
from autogc_validation.database.management import dump_database

dump_database(
    database_path=database,
    output_path=find_project_root() / "data" / "autogc.sql",
)
print("Done. Remember to commit data/autogc.sql.")